In [2]:
import duckdb
con = duckdb.connect("../data/retail.duckdb", read_only=True)
con.sql("SELECT * FROM raw_transactions LIMIT 10").show()

┌─────────┬────────────┬─────────────────────────────────────┬──────────┬─────────────────────┬────────┬─────────────┬────────────────┐
│ invoice │ stock_code │             description             │ quantity │    invoice_date     │ price  │ customer_id │    country     │
│ varchar │  varchar   │               varchar               │  int64   │      timestamp      │ double │   double    │    varchar     │
├─────────┼────────────┼─────────────────────────────────────┼──────────┼─────────────────────┼────────┼─────────────┼────────────────┤
│ 489434  │ 85048      │ 15CM CHRISTMAS GLASS BALL 20 LIGHTS │       12 │ 2009-12-01 07:45:00 │   6.95 │     13085.0 │ United Kingdom │
│ 489434  │ 79323P     │ PINK CHERRY LIGHTS                  │       12 │ 2009-12-01 07:45:00 │   6.75 │     13085.0 │ United Kingdom │
│ 489434  │ 79323W     │  WHITE CHERRY LIGHTS                │       12 │ 2009-12-01 07:45:00 │   6.75 │     13085.0 │ United Kingdom │
│ 489434  │ 22041      │ RECORD FRAME 7" SINGLE 

### Q1: What data range does the data cover? 

In [3]:
con.sql("""SELECT MIN(invoice_date) AS first_date,
        MAX(invoice_date) AS last_date
        FROM raw_transactions""").show()

┌─────────────────────┬─────────────────────┐
│     first_date      │      last_date      │
│      timestamp      │      timestamp      │
├─────────────────────┼─────────────────────┤
│ 2009-12-01 07:45:00 │ 2011-12-09 12:50:00 │
└─────────────────────┴─────────────────────┘



### Q2: How many rows have no customer_id?

In [ ]:
con.sql(""" SELECT COUNT (*) AS total_rows,
        COUNT(*) - COUNT(customer_id) AS null_customer_rows,
        ROUND(100.0 * (COUNT(*) - COUNT(customer_id)) / COUNT(*), 1) AS pct_null
        FROM raw_transactions""").show()

┌────────────┬────────────────────┬──────────┐
│ total_rows │ null_customer_rows │ pct_null │
│   int64    │       int64        │  double  │
├────────────┼────────────────────┼──────────┤
│    1067371 │             243007 │     22.8 │
└────────────┴────────────────────┴──────────┘



Based on the table, 243,007 rows (22.8%) have no customer ID. These are likely due to guest checkouts where customers buy without creating an account. These rows still record real sales, so they count toward revenue, product and country analysis. However, they can't be used for RFM segmentation or cohort retention because both need to know who bought. I will keep them for revenue, product and country analysis but I will exclude them from customer-level analysis. 

### Q3: Some invoice numbers start with a letter instead of a number. Which letters, and what do you think they mean?

In [18]:
con.sql("""SELECT LEFT(invoice, 1) AS invoice_first_char,
        COUNT(*) AS invoice_rows,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM raw_transactions), 3) AS pct_of_all_rows
        FROM raw_transactions
        WHERE NOT regexp_matches(invoice, '^[0-9]')
        GROUP BY LEFT(invoice, 1)
        ORDER BY invoice_first_char""").show()

┌────────────────────┬──────────────┬─────────────────┐
│ invoice_first_char │ invoice_rows │ pct_of_all_rows │
│      varchar       │    int64     │     double      │
├────────────────────┼──────────────┼─────────────────┤
│ A                  │            6 │           0.001 │
│ C                  │        19494 │           1.826 │
└────────────────────┴──────────────┴─────────────────┘



In [24]:
con.sql("""SELECT *
        FROM raw_transactions
        WHERE invoice LIKE 'A%'
        LIMIT 10""").show()

┌─────────┬────────────┬─────────────────┬──────────┬─────────────────────┬───────────┬─────────────┬────────────────┐
│ invoice │ stock_code │   description   │ quantity │    invoice_date     │   price   │ customer_id │    country     │
│ varchar │  varchar   │     varchar     │  int64   │      timestamp      │  double   │   double    │    varchar     │
├─────────┼────────────┼─────────────────┼──────────┼─────────────────────┼───────────┼─────────────┼────────────────┤
│ A506401 │ B          │ Adjust bad debt │        1 │ 2010-04-29 13:36:00 │ -53594.36 │        NULL │ United Kingdom │
│ A516228 │ B          │ Adjust bad debt │        1 │ 2010-07-19 11:24:00 │ -44031.79 │        NULL │ United Kingdom │
│ A528059 │ B          │ Adjust bad debt │        1 │ 2010-10-20 12:04:00 │ -38925.87 │        NULL │ United Kingdom │
│ A563185 │ B          │ Adjust bad debt │        1 │ 2011-08-12 14:50:00 │  11062.06 │        NULL │ United Kingdom │
│ A563186 │ B          │ Adjust bad debt │      

In [23]:
con.sql("""SELECT *
        FROM raw_transactions
        WHERE invoice LIKE 'C%'
        LIMIT 10""").show()

┌─────────┬────────────┬───────────────────────────────────┬──────────┬─────────────────────┬────────┬─────────────┬────────────────┐
│ invoice │ stock_code │            description            │ quantity │    invoice_date     │ price  │ customer_id │    country     │
│ varchar │  varchar   │              varchar              │  int64   │      timestamp      │ double │   double    │    varchar     │
├─────────┼────────────┼───────────────────────────────────┼──────────┼─────────────────────┼────────┼─────────────┼────────────────┤
│ C489449 │ 22087      │ PAPER BUNTING WHITE LACE          │      -12 │ 2009-12-01 10:33:00 │   2.95 │     16321.0 │ Australia      │
│ C489449 │ 85206A     │ CREAM FELT EASTER EGG BASKET      │       -6 │ 2009-12-01 10:33:00 │   1.65 │     16321.0 │ Australia      │
│ C489449 │ 21895      │ POTTING SHED SOW 'N' GROW SET     │       -4 │ 2009-12-01 10:33:00 │   4.25 │     16321.0 │ Australia      │
│ C489449 │ 21896      │ POTTING SHED TWINE                │  

From invoice column, I found two sets of invoices starting from letters. First is invoices starting with 'A' with 6 rows (0.001%) and C with 19,494 rows (1.826%). The A rows are bad debt adjustments (description: 'Adjust bad debt') and the C rows are due to cancellations supported by negative values in quantity. I will exclude A rows from sales analysis because they are accounting adjustments, not product sales and exclude C rows for analyzing sales, but the orders cancelled should be noted, since the cancellation rate is a useful business metric on its own.

### Q4: How many rows have quantity <= 0 or price <= 0?

In [ ]:
con.sql("""SELECT LEFT(invoice, 1) AS invoice_first_char,
        COUNT(*) AS invoice_rows,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM raw_transactions), 3) AS pct_of_all_rows
        FROM raw_transactions
        WHERE NOT regexp_matches(invoice, '^[0-9]')
        GROUP BY LEFT(invoice, 1)
        ORDER BY invoice_first_char""").show()

┌────────────────────┬──────────────┬─────────────────┐
│ invoice_first_char │ invoice_rows │ pct_of_all_rows │
│      varchar       │    int64     │     double      │
├────────────────────┼──────────────┼─────────────────┤
│ A                  │            6 │           0.001 │
│ C                  │        19494 │           1.826 │
└────────────────────┴──────────────┴─────────────────┘



In [61]:
con.sql("""SELECT COUNT (*) AS qty_le_zero
        FROM raw_transactions
        WHERE quantity <= 0
        """).show()

con.sql("""SELECT COUNT (*) AS qty_le_zero_not_c
        FROM raw_transactions
        WHERE quantity <= 0 AND
        invoice NOT LIKE 'C%'
        """).show()

┌─────────────┐
│ qty_le_zero │
│    int64    │
├─────────────┤
│       22950 │
└─────────────┘

┌───────────────────┐
│ qty_le_zero_not_c │
│       int64       │
├───────────────────┤
│              3457 │
└───────────────────┘



Subtraction from qty_le_zero and qty_le_zero_not_c gives 19,493 entries. It is 1 less than the total cancellation count. Probably due to a positive quantity value. 

In [60]:
con.sql("""SELECT *
        FROM raw_transactions
        WHERE quantity <= 0 AND
        invoice NOT LIKE 'C%'
        LIMIT 10
        """).show()

┌─────────┬────────────┬─────────────────┬──────────┬─────────────────────┬────────┬─────────────┬────────────────┐
│ invoice │ stock_code │   description   │ quantity │    invoice_date     │ price  │ customer_id │    country     │
│ varchar │  varchar   │     varchar     │  int64   │      timestamp      │ double │   double    │    varchar     │
├─────────┼────────────┼─────────────────┼──────────┼─────────────────────┼────────┼─────────────┼────────────────┤
│ 489464  │ 21733      │ 85123a mixed    │      -96 │ 2009-12-01 10:52:00 │    0.0 │        NULL │ United Kingdom │
│ 489463  │ 71477      │ short           │     -240 │ 2009-12-01 10:52:00 │    0.0 │        NULL │ United Kingdom │
│ 489467  │ 85123A     │ 21733 mixed     │     -192 │ 2009-12-01 10:53:00 │    0.0 │        NULL │ United Kingdom │
│ 489521  │ 21646      │ NULL            │      -50 │ 2009-12-01 11:44:00 │    0.0 │        NULL │ United Kingdom │
│ 489655  │ 20683      │ NULL            │      -44 │ 2009-12-01 17:26:0

The table shows invoices that were not cancellations but have negative quantity values and null values in customer_id. These are orders that were canceled or modified by a staff member.

In [55]:
con.sql("""SELECT *
        FROM raw_transactions
        WHERE invoice LIKE 'C%'
        AND quantity > 0
        """).show()

┌─────────┬────────────┬─────────────┬──────────┬─────────────────────┬────────┬─────────────┬────────────────┐
│ invoice │ stock_code │ description │ quantity │    invoice_date     │ price  │ customer_id │    country     │
│ varchar │  varchar   │   varchar   │  int64   │      timestamp      │ double │   double    │    varchar     │
├─────────┼────────────┼─────────────┼──────────┼─────────────────────┼────────┼─────────────┼────────────────┤
│ C496350 │ M          │ Manual      │        1 │ 2010-02-01 08:24:00 │ 373.57 │        NULL │ United Kingdom │
└─────────┴────────────┴─────────────┴──────────┴─────────────────────┴────────┴─────────────┴────────────────┘



Table shows the 1 cancellation invoice that has a positive quantity value. Based on stock_code (M) and description (Manual), it is a manual adjustment invoice. Since all C rows are already excluded from sales analysis, this row has no impact on revenue. It is a manual adjustment rather than a cancelled product, but at 1 out of 19,494 rows it does not affect the cancellation rate either.

In [68]:
con.sql("""SELECT COUNT (*) AS price_le_zero
        FROM raw_transactions
        WHERE price <= 0
        """).show()

con.sql("""SELECT COUNT (*) AS price_lt_zero
        FROM raw_transactions
        WHERE price < 0
        """).show()

con.sql("""SELECT COUNT (*) AS price_eq_zero
        FROM raw_transactions
        WHERE price = 0
        """).show()

┌───────────────┐
│ price_le_zero │
│     int64     │
├───────────────┤
│          6207 │
└───────────────┘

┌───────────────┐
│ price_lt_zero │
│     int64     │
├───────────────┤
│             5 │
└───────────────┘

┌───────────────┐
│ price_eq_zero │
│     int64     │
├───────────────┤
│          6202 │
└───────────────┘



In [66]:
con.sql("""SELECT *
        FROM raw_transactions
        WHERE price = 0
        LIMIT 10
        """).show()

┌─────────┬────────────┬──────────────┬──────────┬─────────────────────┬────────┬─────────────┬────────────────┐
│ invoice │ stock_code │ description  │ quantity │    invoice_date     │ price  │ customer_id │    country     │
│ varchar │  varchar   │   varchar    │  int64   │      timestamp      │ double │   double    │    varchar     │
├─────────┼────────────┼──────────────┼──────────┼─────────────────────┼────────┼─────────────┼────────────────┤
│ 489464  │ 21733      │ 85123a mixed │      -96 │ 2009-12-01 10:52:00 │    0.0 │        NULL │ United Kingdom │
│ 489463  │ 71477      │ short        │     -240 │ 2009-12-01 10:52:00 │    0.0 │        NULL │ United Kingdom │
│ 489467  │ 85123A     │ 21733 mixed  │     -192 │ 2009-12-01 10:53:00 │    0.0 │        NULL │ United Kingdom │
│ 489521  │ 21646      │ NULL         │      -50 │ 2009-12-01 11:44:00 │    0.0 │        NULL │ United Kingdom │
│ 489655  │ 20683      │ NULL         │      -44 │ 2009-12-01 17:26:00 │    0.0 │        NULL │ 

In [67]:
con.sql("""SELECT *
        FROM raw_transactions
        WHERE price < 0
        """).show()

┌─────────┬────────────┬─────────────────┬──────────┬─────────────────────┬───────────┬─────────────┬────────────────┐
│ invoice │ stock_code │   description   │ quantity │    invoice_date     │   price   │ customer_id │    country     │
│ varchar │  varchar   │     varchar     │  int64   │      timestamp      │  double   │   double    │    varchar     │
├─────────┼────────────┼─────────────────┼──────────┼─────────────────────┼───────────┼─────────────┼────────────────┤
│ A506401 │ B          │ Adjust bad debt │        1 │ 2010-04-29 13:36:00 │ -53594.36 │        NULL │ United Kingdom │
│ A516228 │ B          │ Adjust bad debt │        1 │ 2010-07-19 11:24:00 │ -44031.79 │        NULL │ United Kingdom │
│ A528059 │ B          │ Adjust bad debt │        1 │ 2010-10-20 12:04:00 │ -38925.87 │        NULL │ United Kingdom │
│ A563186 │ B          │ Adjust bad debt │        1 │ 2011-08-12 14:51:00 │ -11062.06 │        NULL │ United Kingdom │
│ A563187 │ B          │ Adjust bad debt │      

To sum up:

Cancellations (C invoices, quantity<=0): 19,493 rows, same as Q3 (for exclusion in sales analysis)

Inventory adjustments (not C, quantity<=0): 3,457 rows. Came from manual inventory adjustments or modifications made by a staff member evidenced by description such as lost, damages, short, mixed, etc and no customer_id values.

Zero-price: 6,202 rows. These are also inventory modifications where descriptions are made by staff members but quantity values were not all <=0. Positive values possibly due to stock corrections or free items.

Negative-price: 5 rows. These invoices were same as the A rows from Q3 where they are described as bad debt adjustments. However, the A rows contain 6 invoices. That 1 invoice have positive price value. 

Decision: For sales analysis, these cancellation orders, inventory and bad debt adjustments invoices must be excluded, while the orders where quantity > 0 and price > 0 must be kept. 

### Q5: How many rows are exact duplicates?

In [7]:
con.sql("""WITH counts AS (
            SELECT
                (SELECT COUNT(*) FROM raw_transactions) AS total_rows,
                (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM raw_transactions) AS t) AS unique_rows
        )
        SELECT total_rows,
               unique_rows,
               total_rows - unique_rows AS duplicate_rows,
               ROUND(100.0 * (total_rows - unique_rows) / total_rows, 2) AS pct_duplicate
        FROM counts""").show()

┌────────────┬─────────────┬────────────────┬───────────────┐
│ total_rows │ unique_rows │ duplicate_rows │ pct_duplicate │
│   int64    │    int64    │     int64      │    double     │
├────────────┼─────────────┼────────────────┼───────────────┤
│    1067371 │     1033036 │          34335 │          3.22 │
└────────────┴─────────────┴────────────────┴───────────────┘



In [8]:
con.sql("""SELECT invoice, stock_code, description, quantity, invoice_date, price, customer_id, country,
               COUNT(*) AS times_seen
        FROM raw_transactions
        GROUP BY ALL
        HAVING COUNT (*) > 1
        ORDER BY times_seen DESC
        LIMIT 10""").show()

┌─────────┬────────────┬─────────────────────────────────────┬──────────┬─────────────────────┬────────┬─────────────┬────────────────┬────────────┐
│ invoice │ stock_code │             description             │ quantity │    invoice_date     │ price  │ customer_id │    country     │ times_seen │
│ varchar │  varchar   │               varchar               │  int64   │      timestamp      │ double │   double    │    varchar     │   int64    │
├─────────┼────────────┼─────────────────────────────────────┼──────────┼─────────────────────┼────────┼─────────────┼────────────────┼────────────┤
│ 555524  │ 22698      │ PINK REGENCY TEACUP AND SAUCER      │        1 │ 2011-06-05 11:37:00 │   2.95 │     16923.0 │ United Kingdom │         20 │
│ 555524  │ 22697      │ GREEN REGENCY TEACUP AND SAUCER     │        1 │ 2011-06-05 11:37:00 │   2.95 │     16923.0 │ United Kingdom │         12 │
│ 537224  │ 70007      │ HI TEC ALPINE HAND WARMER           │        1 │ 2010-12-05 16:24:00 │   1.65 │  

34,335 rows or 3.22% of all rows are duplicates. These are likely due to a system glitch where lines were copied twice by mistake and/or a staff member entered each item as its own line (for example, invoice 555524 contains the same teacup line 20 times within the same minute). A copy of one of each duplicates should be kept as this can overstate revenue. However, it should be considered that some duplicates may be genuine items entered line by line, so revenue could be slightly understated. Overall, at 3.22% of rows, the effect is only small. 

### Q6: Which stock codes aren't real products? 

In [23]:
con.sql("""SELECT stock_code,
        COUNT (*) AS stock_code_count,
        ANY_VALUE(description) AS ex_desc
        FROM raw_transactions
        WHERE NOT regexp_matches(stock_code, '^[0-9]{5}')
        GROUP BY stock_code
        ORDER BY stock_code_count DESC""").show(max_rows=100)

┌──────────────┬──────────────────┬─────────────────────────────────────┐
│  stock_code  │ stock_code_count │               ex_desc               │
│   varchar    │      int64       │               varchar               │
├──────────────┼──────────────────┼─────────────────────────────────────┤
│ POST         │             2122 │ POSTAGE                             │
│ DOT          │             1446 │ DOTCOM POSTAGE                      │
│ M            │             1421 │ Manual                              │
│ C2           │              282 │ CARRIAGE                            │
│ D            │              177 │ Discount                            │
│ S            │              104 │ SAMPLES                             │
│ BANK CHARGES │              102 │ Bank Charges                        │
│ ADJUST       │               67 │ Adjustment by Peter on 24/05/2010 1 │
│ AMAZONFEE    │               43 │ AMAZON FEE                          │
│ DCGS0058     │               31 │ MI

In [ ]:
con.sql("""SELECT stock_code, description, quantity, price
        FROM raw_transactions
        WHERE stock_code LIKE 'DCGS%'
        AND (description IS NULL OR description = 'ebay')""").show()

┌────────────┬─────────────┬──────────┬────────┐
│ stock_code │ description │ quantity │ price  │
│  varchar   │   varchar   │  int64   │ double │
├────────────┼─────────────┼──────────┼────────┤
│ DCGS0066N  │ NULL        │        1 │    0.0 │
│ DCGS0066N  │ NULL        │        1 │    0.0 │
│ DCGSSGIRL  │ NULL        │       -1 │    0.0 │
│ DCGS0006   │ NULL        │       -1 │    0.0 │
│ DCGS0016   │ NULL        │       -1 │    0.0 │
│ DCGS0027   │ NULL        │       -1 │    0.0 │
│ DCGS0036   │ NULL        │       -1 │    0.0 │
│ DCGS0039   │ NULL        │       -1 │    0.0 │
│ DCGS0060   │ NULL        │       -3 │    0.0 │
│ DCGS0056   │ NULL        │       -4 │    0.0 │
│    ·       │  ·          │        · │     ·  │
│    ·       │  ·          │        · │     ·  │
│    ·       │  ·          │        · │     ·  │
│ DCGS0074   │ NULL        │       -1 │    0.0 │
│ DCGS0057   │ NULL        │       -6 │    0.0 │
│ DCGS0073   │ ebay        │       -4 │    0.0 │
│ DCGS0071   │ NULL 

In [40]:
con.sql("""SELECT stock_code,
               COUNT(*) AS stock_code_count,
               ANY_VALUE(description) AS example_description,
               CASE
                   WHEN stock_code IN ('POST', 'DOT', 'AMAZONFEE', 'M', 'D', 'S', 'BANK CHARGES', 'B', 'CRUK', 'm', 'GIFT', 'C2', 'C3') THEN 'not product'
                   WHEN stock_code LIKE 'gift_%' THEN 'not product'
                   WHEN stock_code LIKE 'TEST%' THEN 'not product'
                   WHEN stock_code LIKE 'ADJUST%' THEN 'not product'
                   ELSE 'product'
               END AS category
        FROM raw_transactions
        WHERE NOT regexp_matches(stock_code, '^[0-9]{5}')
        GROUP BY stock_code
        ORDER BY category DESC, stock_code_count DESC""").show(max_rows=100)


┌──────────────┬──────────────────┬─────────────────────────────────────┬─────────────┐
│  stock_code  │ stock_code_count │         example_description         │  category   │
│   varchar    │      int64       │               varchar               │   varchar   │
├──────────────┼──────────────────┼─────────────────────────────────────┼─────────────┤
│ DCGS0058     │               31 │ MISO PRETTY  GUM                    │ product     │
│ DCGSSGIRL    │               25 │ GIRLS PARTY BAG                     │ product     │
│ DCGSSBOY     │               23 │ BOYS PARTY BAG                      │ product     │
│ PADS         │               19 │ PADS TO MATCH ALL CUSHIONS          │ product     │
│ DCGS0076     │               15 │ SUNJAR LED NIGHT NIGHT LIGHT        │ product     │
│ DCGS0003     │               14 │ BOXED GLASS ASHTRAY                 │ product     │
│ DCGS0069     │                6 │ OOH LA LA DOGS COLLAR               │ product     │
│ DCGS0004     │                

In [48]:
con.sql("""WITH codes AS (
              SELECT stock_code,
              COUNT(*) AS stock_code_count,
              ANY_VALUE(description) AS example_description,
               CASE
                   WHEN stock_code IN ('POST', 'DOT', 'AMAZONFEE', 'M', 'D', 'S', 'BANK CHARGES', 'B', 'CRUK', 'm', 'GIFT', 'C2', 'C3') THEN 'not product'
                   WHEN stock_code LIKE 'gift_%' THEN 'not product'
                   WHEN stock_code LIKE 'TEST%' THEN 'not product'
                   WHEN stock_code LIKE 'ADJUST%' THEN 'not product'
                   ELSE 'product'
               END AS category
              FROM raw_transactions
              WHERE NOT regexp_matches(stock_code, '^[0-9]{5}')
              GROUP BY stock_code
        )
        SELECT category,
               COUNT(*) AS num_codes,
               SUM(stock_code_count) AS num_rows,
               ROUND(100.0 * SUM(stock_code_count) / (SELECT COUNT(*) FROM raw_transactions), 2) AS pct_of_all_rows
        FROM codes
        GROUP BY category""").show()


┌─────────────┬───────────┬──────────┬─────────────────┐
│  category   │ num_codes │ num_rows │ pct_of_all_rows │
│   varchar   │   int64   │  int128  │     double      │
├─────────────┼───────────┼──────────┼─────────────────┤
│ not product │        26 │     5913 │            0.55 │
│ product     │        36 │      180 │            0.02 │
└─────────────┴───────────┴──────────┴─────────────────┘



62 stock codes were found that don't start with 5 digits and contain different formats. 26 of these codes were identified as not real products, covering 5913 rows (0.55%) of all transactions, like stock codes 'POST', 'DOT', and 'M' having the largest counts which are shown as postages, dotcom postage, manual, fees, test codes, samples, bad debt, and vouchers as seen in the description column. Real products with unusual codes include most DCGS codes (e.g. DCGS0003, a boxed glass ashtray), PADS and SP1002. DCGS rows with NULL/ebay descriptions have 0 in price so these will be removed by the Q4 rule. Overall, the 21 non-product stock codes should be excluded from product and sales analysis.

### Q7: Is customer_id stored correctly?

In [49]:
con.sql("DESCRIBE raw_transactions").show()

┌──────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │ column_type │  null   │   key   │ default │  extra  │
│   varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ invoice      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ stock_code   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ description  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ quantity     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ invoice_date │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ price        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id  │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ country      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



customer_id is stored as DOUBLE (a decimal number) so IDs appear as 13085.0 instead of 13085. This likely happened because the column has missing values (from Q2) and when loading the data, pandas stores them as float (shown as DOUBLE in DuckDB). In the cleaning step, I will convert customer_id to VARCHAR because IDs are labels, not numbers. 